# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shenone77/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

A frozen exposure rule, two bucket checks, and ten reviewed picks.

## Setup

Open this notebook from the repository or its Colab badge. The analysis uses FlyRank's bundled anonymized sample.

In [1]:
from pathlib import Path
import subprocess
import sys

if "google.colab" in sys.modules:
    root = Path("/content/flyrank-ml-internship")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/Shenone77/flyrank-ml-internship.git", str(root)],
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas>=2.2", "scikit-learn>=1.4"], check=True)
else:
    start = Path.cwd().resolve()
    root = next(
        (path for path in (start, *start.parents) if (path / "work/notebooks/refresh_triage.py").is_file()),
        None,
    )
    if root is None:
        raise FileNotFoundError("Open the notebook inside a clone of the assignment repository.")

sys.path.insert(0, str(root / "work/notebooks"))
from refresh_triage import load_pages

pages = load_pages(root)
print(f"Loaded {len(pages):,} eligible pages from the FlyRank starter sample.")

Loaded 13,412 eligible pages from the FlyRank starter sample.


## 1. Two signal checks and my rule

**Volume hypothesis.** Higher pre-cutoff impressions may indicate a higher chance of material decline. Volume is the exposure signal FlyRank uses behind its quick-win logic. Here I test volume as its own signal.

**CTR hypothesis.** Pages with low prior CTR may have a higher material-decline rate. CTR is from the preceding 30 days. This is a descriptive bucket comparison without position adjustment. It measures a directional signal for this review rule.

Check both on the training clients. Freeze a hand score equal to prior impressions multiplied by 1.25 when prior CTR is at most 0.5%, or by 1.00 otherwise. Higher prior volume carries more absolute visibility at stake even when it carries a lower observed decline rate. The single reason code `PRIOR_EXPOSURE_REVIEW` means the page has measurable past visibility, with a fixed low-CTR adjustment where applicable. The action is a manual visibility review.

In [2]:
import numpy as np
import pandas as pd
from refresh_triage import (
    LOW_CTR_PCT, REVIEW_BUDGET, baseline_score, client_holdout, precision_at_k, ranked,
)

train, holdout = client_holdout(pages)
checks = [
    (
        "Prior volume",
        pd.cut(train["impressions_prev_30d"], [299, 999, 2999, np.inf],
               labels=["300-999", "1000-2999", "3000+"]),
        True,
    ),
    (
        "Prior CTR (%)",
        pd.cut(train["prior_ctr_pct"], [-1, 0, LOW_CTR_PCT, 2, 100],
               labels=["0", ">0 to 0.5", ">0.5 to 2", ">2"]),
        False,
    ),
]
for name, bins, expected_increase in checks:
    table = train.groupby(bins, observed=True)["material_drop"].agg(n="size", decline_rate="mean")
    rate = table["decline_rate"]
    increasing = rate.is_monotonic_increasing
    decreasing = rate.is_monotonic_decreasing
    if increasing and decreasing:
        verdict = "FALSE"
    elif (expected_increase and increasing) or (not expected_increase and decreasing):
        verdict = "CONFIRMED"
    elif increasing or decreasing:
        verdict = "OPPOSITE"
    else:
        verdict = "MIXED"
    print(f"{name}: {verdict}")
    display(table.assign(decline_pct=rate.mul(100).round(1)).drop(columns="decline_rate"))

Prior volume: OPPOSITE


,n,decline_pct
impressions_prev_30d,,
300-999,5007,49.9
1000-2999,3888,48.7
3000+,3547,40.2


Prior CTR (%): CONFIRMED


,n,decline_pct
prior_ctr_pct,,
0,3175,58.4
>0 to 0.5,7269,46.9
>0.5 to 2,1956,27.7
>2,42,26.2


In the saved run, the high-volume group had a lower observed decline share than the low-volume group. That opposes the volume hypothesis, while the volume term still reflects exposure at stake. The CTR buckets support the low-CTR boost directionally. The highest CTR bucket is small, and client mix might partly explain both patterns.

## 2. Build the ranked queue

Score every eligible row with the frozen rule. Break exact ties using the shared seeded ordering. Keep the queue as a local, regenerable export with pseudonymous keys for lookups. The comparison score below comes from the untouched held-out clients and uses the same ranking policy as ML-08.

In [3]:
queue = ranked(pages, baseline_score(pages)).assign(
    reason_code="PRIOR_EXPOSURE_REVIEW",
    action="Review search visibility",
)
queue.insert(0, "rank", np.arange(1, len(queue) + 1))
output = root / "work/outputs/baseline_action_score.csv"
output.parent.mkdir(parents=True, exist_ok=True)
export_columns = [
    "rank", "client_id", "content_id", "impressions_prev_30d",
    "prior_ctr_pct", "score", "reason_code", "action",
]
queue[export_columns].to_csv(output, index=False)
print(f"Wrote {len(queue):,} ranked rows to work/outputs/baseline_action_score.csv.")
rule_test = ranked(holdout, baseline_score(holdout))
print(f"Held-out baseline Precision@{REVIEW_BUDGET}: {precision_at_k(rule_test):.1%}")
print(f"Held-out material-drop base rate: {holdout['material_drop'].mean():.1%}")

Wrote 13,412 ranked rows to work/outputs/baseline_action_score.csv.
Held-out baseline Precision@50: 84.0%
Held-out material-drop base rate: 85.3%


## 3. Top-10 review

Each row is a real pick from the full candidate queue. The public table rounds exposure and age and omits keys. The action in each row is a check an editor can take. The last column gives a concrete condition that would make the suggested review less useful.

In [4]:
top = queue.head(10)
review = pd.DataFrame({
    "Rank": top["rank"],
    "Prior impressions (thousands)": top["impressions_prev_30d"].div(1000).round(),
    "Prior CTR (%)": top["prior_ctr_pct"].round(2),
    "Age at cutoff (30-day band)": top["age_at_cutoff_days"].floordiv(30).mul(30),
    "Action": top["action"],
    "Why ranked": np.where(
        top["prior_ctr_pct"].le(LOW_CTR_PCT), "Strong exposure + low CTR boost", "Strong exposure alone"
    ),
    "What would make it wrong": [
        "Exposure came from a documented one-off demand spike.",
        "The page already has an approved migration or replacement.",
        "A broad query mix explains the low CTR without a page-level issue.",
        "A younger page is still settling after its launch promotion.",
        "The score is inflated by impressions from markets the editor does not serve.",
        "The higher CTR and stable clicks make visibility review a lower priority.",
        "The page competes with a preferred canonical that the editor should check first.",
        "The CTR sits near the boost cutoff, and a small measurement change could remove the boost.",
        "A short history and seasonal demand make the high exposure hard to repeat.",
        "Near-zero CTR reflects irrelevant queries, so a refresh would target the wrong problem.",
    ],
})
display(review.reset_index(drop=True))
assert len(review) == 10 and review["What would make it wrong"].nunique() == 10

,Rank,Prior impressions (thousands),Prior CTR (%),Age at cutoff (30-day band),Action,Why ranked,What would make it wrong
0,1,219.0,0.11,480,Review search visibility,Strong exposure + low CTR boost,Exposure came from a documented one-off demand...
1,2,178.0,0.26,390,Review search visibility,Strong exposure + low CTR boost,The page already has an approved migration or ...
2,3,161.0,0.24,390,Review search visibility,Strong exposure + low CTR boost,A broad query mix explains the low CTR without...
3,4,161.0,0.08,120,Review search visibility,Strong exposure + low CTR boost,A younger page is still settling after its lau...
4,5,148.0,0.22,450,Review search visibility,Strong exposure + low CTR boost,The score is inflated by impressions from mark...
5,6,174.0,0.83,390,Review search visibility,Strong exposure alone,The higher CTR and stable clicks make visibili...
6,7,138.0,0.23,240,Review search visibility,Strong exposure + low CTR boost,The page competes with a preferred canonical t...
7,8,164.0,0.52,330,Review search visibility,Strong exposure alone,"The CTR sits near the boost cutoff, and a smal..."
8,9,124.0,0.14,90,Review search visibility,Strong exposure + low CTR boost,A short history and seasonal demand make the h...
9,10,112.0,0.00,330,Review search visibility,Strong exposure + low CTR boost,"Near-zero CTR reflects irrelevant queries, so ..."


## 4. Weak picks and leakage check

Rank 6 entered on exposure alone. Its CTR is stronger than most picks, so I would ask whether visibility actually puts useful clicks at risk before spending an edit slot. Rank 8 sits near the fixed CTR threshold, which makes its position sensitive to a small change in observed CTR. These are review questions; the snapshot does not establish those explanations.

Both signals in the score come from the preceding 30 days. The follow-up window appears only in the bucket outcomes and evaluation. Outcome columns and IDs are omitted from the ranking inputs and outcome columns are omitted from the exported queue.

In [5]:
assert queue["score"].is_monotonic_decreasing
assert queue["reason_code"].nunique() == 1
assert "material_drop" not in export_columns and "impressions_last_30d" not in export_columns
assert output.is_file()
print("Self-check passed: descending queue, single reason code, ten reviews, and regenerable CSV.")

Self-check passed: descending queue, single reason code, ten reviews, and regenerable CSV.
